# TableGuard-Lite · Phase 2

Run this notebook inside the SAME extracted `TableGuard_Starter` project after applying the update. This phase inspects a real scene and captures RGB. It does not contain a pretrained robot policy or claim a completed task.

In [ ]:
from pathlib import Path
import sys, subprocess, json

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "tableguard/phase2").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Apply the Phase 2 update inside TableGuard_Starter first.")
print("Project:", ROOT)
print("Python:", sys.executable)

def run_phase2(*args, allow_incomplete=False):
    result = subprocess.run([sys.executable, "-m", "tableguard.phase2", *map(str, args)], cwd=ROOT, text=True, capture_output=True)
    print(result.stdout)
    if result.stderr:
        print(result.stderr)
    if result.returncode and not allow_incomplete:
        raise RuntimeError(f"Command exited {result.returncode}; inspect the error above.")
    return result.returncode

## 1. Current-environment preflight
No package, driver, policy or dataset is downloaded. Exit code 2 can indicate missing scene dependencies. Use the organizer-compatible environment rather than blindly upgrading a working stack.

In [ ]:
run_phase2("preflight", allow_incomplete=True)
report = json.loads((ROOT / "artifacts/phase2_preflight.json").read_text())
print("Missing:", report["missing_scene_dependencies"])

## 2. Offline regression tests
The optional four MuJoCo API tests are skipped unless explicitly enabled after installation. Passing these software tests is not a completed robot episode.

In [ ]:
run_phase2("selftest")

## 3. Scan only the permitted asset folder
Extract the complete task assets first. Do not replace missing task assets with the test fixture or an unrelated public dataset.

In [ ]:
ASSET_DIR = ROOT / "assets/challenge"
run_phase2("scan", "--directory", ASSET_DIR)

## 4. Set the actual task entrypoint
This cell is intentionally skipped until you set a real file. Keep its relative meshes/includes intact. When the organizer supplies a Python environment API instead of a standalone scene, use that API for task execution.

In [ ]:
SCENE = None  # Example syntax only: ROOT / "assets/challenge/<actual entrypoint>.xml"

if SCENE is None:
    print("Set SCENE to the actual permitted MJCF entrypoint before continuing.")
else:
    SCENE = Path(SCENE)
    if not SCENE.is_file():
        raise FileNotFoundError(SCENE)
    run_phase2("inspect", "--scene", SCENE)

## 5. Capture and display actual camera images
Set GL_BACKEND only as required by the machine. Defaults use the platform renderer. This capture does not run a policy or step physics.

In [ ]:
GL_BACKEND = None  # Linux headless examples: "egl" or "osmesa", only with the required libraries.
if SCENE is None:
    print("Skipped: the actual scene is not configured.")
else:
    args = ["capture", "--scene", SCENE]
    if GL_BACKEND:
        args += ["--gl", GL_BACKEND]
    run_phase2(*args)
    pointer = json.loads((ROOT / "artifacts/phase2_latest_capture.json").read_text())
    capture = json.loads(Path(pointer["report_path"]).read_text())
    from IPython.display import display, Image
    for image in capture["images"]:
        print(image["camera"])
        display(Image(filename=image["file"]))
    print("Task success:", capture["task_success"], "(not evaluated)")

## 6. Create the explicit policy contract
The template deliberately contains unconfigured fields. Populate them from real documentation. Do not flip permissions to true merely to bypass validation. Policy adaptation is in `integrations/approved_policy_template.py`; no pretrained model is included.

In [ ]:
CONFIG = ROOT / "configs/phase2_task_contract.json"
if SCENE is None:
    print("Skipped: configure the real scene first.")
elif CONFIG.exists():
    print("Existing contract preserved:", CONFIG)
else:
    run_phase2("init", "--scene", SCENE)
if CONFIG.exists():
    run_phase2("check", allow_incomplete=True)

## Next handoff
Share the preflight JSON, scene inventory, camera report, and actual brief/scene/policy instructions. Keep API tokens and other credentials private. Do not run the policy command until the real adapter is implemented and reviewed. Read `README_PHASE2.md` for the exact input and native-control contract.